# SCADA Lift-Station KPI Calculator
40 KPIs (KPI_001 – KPI_040) · 7 days × 1-min interval mock SCADA · pure Python stdlib

In [1]:
import random, math, json, statistics, datetime
from collections import defaultdict
random.seed(2025)

# ── Station constants ──────────────────────────────────────────────────────
STATION_ID   = 'LS_001'
WW_AREA_SQFT = 78.54          # circular wet well, ~10 ft diameter
WW_DEPTH_MAX = 18.0
WW_DEPTH_MIN = 2.0
PUMP_IDS     = ['P1', 'P2', 'P3']
PUMP_ROLES   = {'P1': 'duty', 'P2': 'lag', 'P3': 'assist'}
PUMP_RATING_A= {'P1': 125.0, 'P2': 120.0, 'P3': 118.0}
# Flow rates sized so P1 barely handles dry-weather; P2 needed for moderate rain
PUMP_FLOW_GPM= {'P1': 270.0, 'P2': 265.0, 'P3': 260.0}

# Pump setpoints – levels are within the reachable 5–14 ft operating band
PUMP_ON_LEVEL = {'P1': 11.0, 'P2': 12.5, 'P3': 14.0}
PUMP_OFF_LEVEL= {'P1':  5.5, 'P2':  7.0, 'P3':  8.5}

SHORT_RUN_THRESH = 4.0        # minutes
LONG_RUN_THRESH  = 45.0       # minutes
STRESS_LEVEL_FT  = 14.5       # high wet-well alarm threshold

SIM_DAYS  = 7
N_MINUTES = SIM_DAYS * 1440
T0        = datetime.datetime(2025, 5, 1, 0, 0, 0)

# Rain events: (start_min, duration_min, peak_intensity_in_per_5min)
# Day 2 00:00 → moderate 3-hr;  Day 5 18:00 → heavy 5-hr
RAIN_EVENTS_DEF = [
    (2160, 180, 0.08),
    (7020, 300, 0.14),
]

print(f'Simulation: {N_MINUTES:,} minutes ({SIM_DAYS} days), station {STATION_ID}')
print(f'Pump flows: {PUMP_FLOW_GPM}')
print(f'Pump ON setpoints: {PUMP_ON_LEVEL}')

Simulation: 10,080 minutes (7 days), station LS_001
Pump flows: {'P1': 270.0, 'P2': 265.0, 'P3': 260.0}
Pump ON setpoints: {'P1': 11.0, 'P2': 12.5, 'P3': 14.0}


## 1. Generate 1-Minute SCADA Time Series

In [2]:
def rain_rate_at(t_min):
    total = 0.0
    for start, dur, peak in RAIN_EVENTS_DEF:
        if start <= t_min < start + dur:
            phase = (t_min - start) / dur
            intensity = peak * math.exp(-8 * (phase - 0.5)**2)
            total += intensity / 5.0
    return total

def dry_inflow_gpm(t_min):
    """Diurnal dry-weather inflow: 100–230 gpm (slightly below P1 capacity)."""
    hour_frac = (t_min % 1440) / 60.0
    base = 155.0
    morning = 55 * math.exp(-0.5 * ((hour_frac - 7) / 1.5)**2)
    evening = 40 * math.exp(-0.5 * ((hour_frac - 20) / 2.0)**2)
    noise   = random.gauss(0, 5)
    return max(40.0, base + morning + evening + noise)

def rain_inflow_bonus(t_min):
    """I&I inflow bonus during/after rain.
    Peak bonus ≈ 500 gpm (moderate) to 900 gpm (heavy) → pushes past P1 capacity."""
    bonus = 0.0
    for start, dur, peak in RAIN_EVENTS_DEF:
        if t_min >= start:
            elapsed = t_min - start
            decay_end = dur * 3
            if elapsed < decay_end:
                # multiplier 7000 → moderate: 0.08*7000=560; heavy: 0.14*7000=980 gpm peak
                bonus += peak * 7000 * math.exp(-elapsed / (dur * 0.9))
    return bonus

def amp_noise(base, on):
    if not on: return 0.0
    return round(base + random.gauss(0, 2.5), 1)

# ── State machine simulation ───────────────────────────────────────────────
GAL_PER_CUFT = 7.4805
level = 8.0
pump_state = {p: False for p in PUMP_IDS}

records = []

for t in range(N_MINUTES):
    ts = T0 + datetime.timedelta(minutes=t)
    q_in = dry_inflow_gpm(t) + rain_inflow_bonus(t)
    rain  = rain_rate_at(t)

    # Pump control: turn on if level ≥ setpoint; turn off if level ≤ off setpoint
    for p in PUMP_IDS:
        if not pump_state[p] and level >= PUMP_ON_LEVEL[p]:
            pump_state[p] = True
        elif pump_state[p] and level <= PUMP_OFF_LEVEL[p]:
            pump_state[p] = False

    q_out = sum(PUMP_FLOW_GPM[p] for p in PUMP_IDS if pump_state[p])
    n_on  = sum(1 for p in PUMP_IDS if pump_state[p])

    delta_level = (q_in - q_out) / (WW_AREA_SQFT * GAL_PER_CUFT)
    level = max(WW_DEPTH_MIN, min(WW_DEPTH_MAX, level + delta_level))

    rec = {
        'ts': ts.strftime('%Y-%m-%d %H:%M:%S'),
        'minute': t,
        'date':   ts.strftime('%Y-%m-%d'),
        'day':    ts.weekday(),
        'ww_level_ft':    round(level, 3),
        'q_inflow_gpm':   round(q_in, 1),
        'rain_in_per_min': round(rain, 5),
        'n_pumps_on':     n_on,
    }
    for p in PUMP_IDS:
        rec[f'{p}_on']  = 1 if pump_state[p] else 0
        rec[f'{p}_amp'] = amp_noise(PUMP_RATING_A[p] * random.uniform(0.82, 0.96), pump_state[p])
    records.append(rec)

print(f'Generated {len(records):,} SCADA records')
print(f'Level range: {min(r["ww_level_ft"] for r in records):.2f} – {max(r["ww_level_ft"] for r in records):.2f} ft')
for p in PUMP_IDS:
    on_mins = sum(r[f'{p}_on'] for r in records)
    print(f'  {p} on-minutes: {on_mins:,}  ({on_mins/N_MINUTES*100:.1f}%)')

Generated 10,080 SCADA records
Level range: 5.30 – 18.00 ft
  P1 on-minutes: 6,862  (68.1%)
  P2 on-minutes: 571  (5.7%)
  P3 on-minutes: 210  (2.1%)


## 2. Eventize Pump Runs

In [3]:
# Detect 0→1 and 1→0 transitions per pump; build run events
run_events = []   # {pump, start_min, end_min, duration_min, start_level, end_level, median_amp, max_amp, date}

for p in PUMP_IDS:
    prev = 0
    run_start = None
    start_level = None
    amps_in_run = []

    for r in records:
        cur = r[f'{p}_on']
        if prev == 0 and cur == 1:          # rising edge → start
            run_start   = r['minute']
            start_level = r['ww_level_ft']
            amps_in_run = []
        if cur == 1 and run_start is not None:
            amps_in_run.append(r[f'{p}_amp'])
        if prev == 1 and cur == 0 and run_start is not None:   # falling edge → end
            end_min  = r['minute']
            dur      = end_min - run_start
            if dur > 0:
                sorted_a = sorted(amps_in_run)
                mid      = len(sorted_a) // 2
                med_amp  = sorted_a[mid] if sorted_a else 0.0
                run_events.append({
                    'pump':       p,
                    'start_min':  run_start,
                    'end_min':    end_min,
                    'duration_min': dur,
                    'start_level': start_level,
                    'end_level':   r['ww_level_ft'],
                    'median_amp':  round(med_amp, 2),
                    'max_amp':     round(max(amps_in_run), 2) if amps_in_run else 0.0,
                    'date':        records[run_start]['date'],
                })
            run_start = None
            amps_in_run = []
        prev = cur

print(f'Total run events: {len(run_events)}')
for p in PUMP_IDS:
    pe = [e for e in run_events if e['pump'] == p]
    print(f'  {p}: {len(pe)} runs, total runtime {sum(e["duration_min"] for e in pe):,} min')

Total run events: 173
  P1: 162 runs, total runtime 6,840 min
  P2: 9 runs, total runtime 571 min
  P3: 2 runs, total runtime 210 min


## 3. Identify Rain Events

In [4]:
# Build rain event windows from RAIN_EVENTS_DEF
# Add 60-min pre-event and up to 4-hr post-event for response tracking
rain_events = []
for idx, (start_min, dur_min, peak) in enumerate(RAIN_EVENTS_DEF):
    end_min      = start_min + dur_min
    event_recs   = [r for r in records if start_min <= r['minute'] < end_min]
    post_recs    = [r for r in records if end_min   <= r['minute'] < end_min + 240]
    pre_recs     = [r for r in records if start_min - 60 <= r['minute'] < start_min]

    rain_total   = sum(r['rain_in_per_min'] * 5 for r in event_recs)   # approx inches
    peak_ww      = max((r['ww_level_ft'] for r in event_recs), default=0)
    event_starts = sum(1 for e in run_events if start_min <= e['start_min'] < end_min)
    event_rt_min = sum(e['duration_min'] for e in run_events if start_min <= e['start_min'] < end_min)
    event_hrs    = dur_min / 60.0

    base_starts  = sum(1 for e in run_events if start_min - 1440 <= e['start_min'] < start_min)
    base_hrs     = 24.0
    base_rt_min  = sum(e['duration_min'] for e in run_events if start_min - 1440 <= e['start_min'] < start_min)

    # KPI_028: lag from rain start to first ww rise of > 0.3 ft
    baseline_lvl = statistics.mean(r['ww_level_ft'] for r in pre_recs) if pre_recs else 9.0
    lag_min = None
    for r in event_recs:
        if r['ww_level_ft'] > baseline_lvl + 0.3:
            lag_min = r['minute'] - start_min
            break
    if lag_min is None: lag_min = dur_min

    # KPI_029: recovery time – minutes after rain end until level drops back to baseline+0.5
    recovery_hr = None
    for r in post_recs:
        if r['ww_level_ft'] <= baseline_lvl + 0.5:
            recovery_hr = round((r['minute'] - end_min) / 60.0, 2)
            break
    if recovery_hr is None: recovery_hr = 4.0

    # KPI_031: fraction of event time with >=2 pumps on
    two_pump_mins = sum(1 for r in event_recs if r['n_pumps_on'] >= 2)
    multi_pump_frac = round(two_pump_mins / max(1, len(event_recs)), 3)

    # KPI_032: assist pump (P3) starts and runtime during event
    p3_event_starts = sum(1 for e in run_events if e['pump'] == 'P3' and start_min <= e['start_min'] < end_min)
    p3_event_rt     = sum(e['duration_min'] for e in run_events if e['pump'] == 'P3' and start_min <= e['start_min'] < end_min)

    rain_events.append({
        'event_id':            f'RE_{idx+1:02d}',
        'start_min':           start_min,
        'end_min':             end_min,
        'duration_min':        dur_min,
        'rain_total_in':       round(rain_total, 3),
        'peak_intensity':      peak,
        'peak_ww_level_ft':    round(peak_ww, 2),
        'event_starts':        event_starts,
        'event_rt_min':        event_rt_min,
        'event_starts_per_hr': round(event_starts / event_hrs, 2),
        'event_rt_per_hr':     round(event_rt_min / event_hrs, 2),
        'baseline_starts_per_hr': round(base_starts / base_hrs, 4),
        'baseline_rt_per_hr':     round(base_rt_min / base_hrs, 2),
        'lag_min':             lag_min,
        'recovery_hr':         recovery_hr,
        'multi_pump_frac':     multi_pump_frac,
        'p3_event_starts':     p3_event_starts,
        'p3_event_rt_min':     p3_event_rt,
    })

for re in rain_events:
    print(f"{re['event_id']}: {re['rain_total_in']:.2f}in, peak WW={re['peak_ww_level_ft']}ft, "
          f"lag={re['lag_min']}min, recovery={re['recovery_hr']}hr")

RE_01: 8.61in, peak WW=14.04ft, lag=0min, recovery=0.55hr
RE_02: 25.12in, peak WW=18.0ft, lag=1min, recovery=0.35hr


## 4. Daily Aggregation

In [5]:
DATES = sorted(set(r['date'] for r in records))

def pct(vals, p):
    if not vals: return 0.0
    sv = sorted(vals)
    idx = (p / 100) * (len(sv) - 1)
    lo, hi = int(idx), min(int(idx)+1, len(sv)-1)
    return sv[lo] + (idx - lo) * (sv[hi] - sv[lo])

daily = {}   # date → kpi dict

for date in DATES:
    day_recs = [r for r in records if r['date'] == date]
    levels   = [r['ww_level_ft'] for r in day_recs]
    total_min = len(day_recs)

    # KPI_001-003
    kpi001 = round(max(levels), 3)
    kpi002 = round(pct(levels, 95), 3)
    kpi003 = round(statistics.mean(levels), 3)

    # KPI_004-005: total starts/runtime all pumps
    day_runs_all = [e for e in run_events if e['date'] == date]
    kpi004 = len(day_runs_all)
    kpi005 = sum(e['duration_min'] for e in day_runs_all)

    # KPI_017: any pump on fraction
    any_on = sum(1 for r in day_recs if r['n_pumps_on'] > 0)
    kpi017 = round(any_on / total_min, 4)

    # KPI_018: two-pump fraction
    two_on = sum(1 for r in day_recs if r['n_pumps_on'] >= 2)
    kpi018 = round(two_on / total_min, 4)

    # KPI_021: fill rate when all pumps off
    off_deltas = []
    for i in range(1, len(day_recs)):
        if day_recs[i]['n_pumps_on'] == 0 and day_recs[i-1]['n_pumps_on'] == 0:
            off_deltas.append(day_recs[i]['ww_level_ft'] - day_recs[i-1]['ww_level_ft'])
    kpi021 = round(statistics.mean(off_deltas), 5) if off_deltas else 0.0

    # KPI_022: inferred inflow when pumps off (gpm)
    inflow_gpm_list = []
    for i in range(1, len(day_recs)):
        if day_recs[i]['n_pumps_on'] == 0:
            dl = day_recs[i]['ww_level_ft'] - day_recs[i-1]['ww_level_ft']
            gpm = (WW_AREA_SQFT * dl * GAL_PER_CUFT) / 1.0   # dt=1 min
            if gpm > 0:
                inflow_gpm_list.append(round(gpm, 2))
    kpi022_mean = round(statistics.mean(inflow_gpm_list), 1) if inflow_gpm_list else 0.0
    kpi023      = round(pct(inflow_gpm_list, 95), 1) if inflow_gpm_list else 0.0

    # KPI_024: rain total
    kpi024 = round(sum(r['rain_in_per_min'] * 1 for r in day_recs), 4)

    daily[date] = {
        'date': date,
        'KPI_001': kpi001, 'KPI_002': kpi002, 'KPI_003': kpi003,
        'KPI_004': kpi004, 'KPI_005': kpi005,
        'KPI_017': kpi017, 'KPI_018': kpi018,
        'KPI_021': kpi021, 'KPI_022': kpi022_mean, 'KPI_023': kpi023,
        'KPI_024': kpi024,
    }

print('Daily KPIs computed for', len(daily), 'days')
print(f"{'Date':<12} {'MaxWW':>7} {'P95WW':>7} {'Starts':>7} {'RT(min)':>8} {'AnyOn%':>8} {'Rain(in)':>9}")
for d in DATES:
    kd = daily[d]
    print(f"{d:<12} {kd['KPI_001']:>7.2f} {kd['KPI_002']:>7.2f} "
          f"{kd['KPI_004']:>7} {kd['KPI_005']:>8} "
          f"{kd['KPI_017']*100:>7.1f}% {kd['KPI_024']:>9.3f}")

Daily KPIs computed for 7 days
Date           MaxWW   P95WW  Starts  RT(min)   AnyOn%  Rain(in)
2025-05-01     11.30   10.87      26      910    63.2%     0.000
2025-05-02     14.04   11.04      23     1248    74.9%     1.723
2025-05-03     11.26   10.83      27      931    64.0%     0.000
2025-05-04     11.32   10.83      26      906    63.6%     0.000
2025-05-05     18.00   18.00      25     2068    68.3%     3.319
2025-05-06     13.60   12.36      21      683    78.4%     1.705
2025-05-07     11.25   10.83      25      875    64.2%     0.000


## 5. Per-Pump Daily KPIs (006–016)

In [6]:
GAL_PER_CUFT = 7.4805
pump_daily = {}   # (date, pump) → kpi dict

for date in DATES:
    for p in PUMP_IDS:
        runs = [e for e in run_events if e['date'] == date and e['pump'] == p]
        durs = [e['duration_min'] for e in runs]

        kpi006 = len(runs)
        kpi007 = sum(durs)
        kpi008 = round(statistics.median(durs), 2) if durs else 0.0
        kpi009 = round(sum(1 for d in durs if d <= SHORT_RUN_THRESH) / max(1, len(durs)), 4)
        kpi010 = round(sum(1 for d in durs if d >= LONG_RUN_THRESH)  / max(1, len(durs)), 4)

        amps = [e['median_amp'] for e in runs if e['median_amp'] > 0]
        max_amps = [e['max_amp'] for e in runs if e['max_amp'] > 0]
        kpi011 = round(statistics.median(amps), 2) if amps else 0.0
        kpi012 = round(max(max_amps), 2) if max_amps else 0.0
        kpi013 = round(kpi011 / PUMP_RATING_A[p], 4) if kpi011 > 0 else 0.0

        drawdowns = [e['start_level'] - e['end_level'] for e in runs]
        drawdown_rates = []
        for e in runs:
            dd = e['start_level'] - e['end_level']
            if e['duration_min'] > 0:
                drawdown_rates.append(dd / e['duration_min'])
        kpi014 = round(statistics.mean(drawdowns), 3) if drawdowns else 0.0
        kpi015 = round(statistics.mean(drawdown_rates), 5) if drawdown_rates else 0.0

        kpi016_list = []
        for e in runs:
            dd  = e['start_level'] - e['end_level']
            denom = e['median_amp'] * e['duration_min']
            if denom > 0 and e['duration_min'] >= 2:
                kpi016_list.append(dd / denom)
        kpi016 = round(statistics.mean(kpi016_list), 6) if kpi016_list else 0.0

        pump_daily[(date, p)] = {
            'date': date, 'pump': p,
            'KPI_006': kpi006, 'KPI_007': kpi007,
            'KPI_008': kpi008, 'KPI_009': kpi009, 'KPI_010': kpi010,
            'KPI_011': kpi011, 'KPI_012': kpi012, 'KPI_013': kpi013,
            'KPI_014': kpi014, 'KPI_015': kpi015, 'KPI_016': kpi016,
        }

print('Pump-daily KPI sample (P1, first 3 days):')
print(f"{'Date':<12}{'Pump':>5}{'Starts':>7}{'Runtime':>8}{'Median':>7}{'ShortF':>7}{'LongF':>7}{'MedA':>7}{'DD/run':>8}")
for date in DATES[:3]:
    kd = pump_daily[(date,'P1')]
    print(f"{date:<12}{'P1':>5}{kd['KPI_006']:>7}{kd['KPI_007']:>8}{kd['KPI_008']:>7.1f}"
          f"{kd['KPI_009']:>7.3f}{kd['KPI_010']:>7.3f}{kd['KPI_011']:>7.1f}{kd['KPI_014']:>8.2f}")

Pump-daily KPI sample (P1, first 3 days):
Date         Pump Starts Runtime Median ShortF  LongF   MedA  DD/run
2025-05-01     P1     26     910   32.0  0.000  0.115  111.4    5.28
2025-05-02     P1     19    1078   33.0  0.000  0.263  111.5    5.26
2025-05-03     P1     27     931   32.0  0.000  0.074  111.6    5.23


## 6. Station-Level & Pump Availability KPIs (019-020)

In [7]:
# KPI_019: Pump availability (monthly) – did pump participate?
kpi019 = {p: (1 if any(e['pump'] == p for e in run_events) else 0) for p in PUMP_IDS}
print('KPI_019 Pump Availability:', kpi019)

# KPI_020: Duty imbalance ratio (weekly) – max runtime share
total_rt = sum(e['duration_min'] for e in run_events)
pump_rt_share = {p: sum(e['duration_min'] for e in run_events if e['pump'] == p) / total_rt
                 for p in PUMP_IDS}
kpi020 = round(max(pump_rt_share.values()), 4)
print('KPI_020 Duty Imbalance Ratio:', kpi020)
print('  Runtime shares:', {p: round(v, 3) for p, v in pump_rt_share.items()})

KPI_019 Pump Availability: {'P1': 1, 'P2': 1, 'P3': 1}
KPI_020 Duty Imbalance Ratio: 0.8975
  Runtime shares: {'P1': 0.898, 'P2': 0.075, 'P3': 0.028}


## 7. Rain-Event KPIs (025-032)

In [8]:
for re in rain_events:
    bsph = re['baseline_starts_per_hr']
    brph = re['baseline_rt_per_hr']
    esph = re['event_starts_per_hr']
    erph = re['event_rt_per_hr']

    re['KPI_025'] = re['rain_total_in']
    re['KPI_026'] = round(esph / bsph, 3) if bsph > 0 else esph
    re['KPI_027'] = round(erph / brph, 3) if brph > 0 else erph
    re['KPI_028'] = re['lag_min']
    re['KPI_029'] = re['recovery_hr']
    re['KPI_030'] = re['peak_ww_level_ft']
    re['KPI_031'] = re['multi_pump_frac']
    re['KPI_032_starts'] = re['p3_event_starts']
    re['KPI_032_rt_min'] = re['p3_event_rt_min']

print('Rain Event KPIs:')
for re in rain_events:
    print(f"  {re['event_id']}: rain={re['KPI_025']:.2f}in "
          f"uplift_starts={re['KPI_026']:.2f}x uplift_rt={re['KPI_027']:.2f}x "
          f"lag={re['KPI_028']}min recovery={re['KPI_029']}hr "
          f"peak_ww={re['KPI_030']}ft 2pump_frac={re['KPI_031']:.2f}")

Rain Event KPIs:
  RE_01: rain=8.61in uplift_starts=0.89x uplift_rt=0.98x lag=0min recovery=0.55hr peak_ww=14.04ft 2pump_frac=0.73
  RE_02: rain=25.12in uplift_starts=0.37x uplift_rt=1.49x lag=1min recovery=0.35hr peak_ww=18.0ft 2pump_frac=0.98


## 8. Dry-Weather Assist Pump & Monthly KPIs (033-040)

In [9]:
# KPI_033: P3 (assist) use on dry-weather days (rain < 0.05 in)
rain_event_mins = set()
for start, dur, _ in RAIN_EVENTS_DEF:
    for m in range(start, start + dur + 120):   # include 2-hr tail
        rain_event_mins.add(m)
dry_days = [d for d in DATES if daily[d]['KPI_024'] < 0.05]
p3_dry_starts = sum(pump_daily[(d,'P3')]['KPI_006'] for d in dry_days)
p3_dry_rt     = sum(pump_daily[(d,'P3')]['KPI_007'] for d in dry_days)
kpi033 = {'starts_per_day': round(p3_dry_starts / max(1, len(dry_days)), 2),
           'rt_per_day':     round(p3_dry_rt     / max(1, len(dry_days)), 1)}
print('KPI_033 Dry-weather P3 use:', kpi033)

# KPI_034: High-WW days count (monthly)
kpi034 = sum(1 for d in DATES if daily[d]['KPI_001'] > STRESS_LEVEL_FT)
print(f'KPI_034 High-WW days (>{STRESS_LEVEL_FT}ft): {kpi034}')

# KPI_035: Short-cycling days per pump
SHORT_CYCLE_FRAC_THRESH = 0.30
kpi035 = {p: sum(1 for d in DATES if pump_daily[(d,p)]['KPI_009'] > SHORT_CYCLE_FRAC_THRESH)
           for p in PUMP_IDS}
print('KPI_035 Short-cycling days:', kpi035)

# KPI_036: Rain events with uplift trigger (starts or runtime uplift > 1.5x)
kpi036 = sum(1 for re in rain_events if re['KPI_026'] > 1.5 or re['KPI_027'] > 1.5)
print(f'KPI_036 Rain stress events: {kpi036}')

# KPI_037: Trigger counts by severity
# Synthesise trigger counts from KPI violations
triggers = []
for d in DATES:
    kd = daily[d]
    if kd['KPI_001'] > STRESS_LEVEL_FT:
        triggers.append({'date': d, 'kpi': 'KPI_001', 'severity': 'HIGH'})
    for p in PUMP_IDS:
        pd = pump_daily[(d, p)]
        if pd['KPI_009'] > 0.35:
            triggers.append({'date': d, 'kpi': 'KPI_009', 'pump': p, 'severity': 'MEDIUM'})
        if pd['KPI_013'] < 0.72 or pd['KPI_013'] > 0.98:
            triggers.append({'date': d, 'kpi': 'KPI_013', 'pump': p, 'severity': 'LOW'})
for re in rain_events:
    if re['KPI_026'] > 2.0:
        triggers.append({'date': re['event_id'], 'kpi': 'KPI_026', 'severity': 'HIGH'})
    if re['KPI_029'] > 2.5:
        triggers.append({'date': re['event_id'], 'kpi': 'KPI_029', 'severity': 'MEDIUM'})

sev_counts = defaultdict(int)
for t in triggers:
    sev_counts[t['severity']] += 1
kpi037 = dict(sev_counts)
print('KPI_037 Trigger counts by severity:', kpi037)

# KPI_038: Trigger persistence index (weighted repeated triggers)
kpi_trigger_counts = defaultdict(int)
for t in triggers:
    kpi_trigger_counts[t['kpi']] += 1
sev_weight = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
kpi038 = sum(sev_weight.get(t['severity'], 1) * kpi_trigger_counts[t['kpi']]
             for t in triggers)
print(f'KPI_038 Trigger Persistence Index: {kpi038}')

# KPI_039: Resilience concern flag
# Criteria: ≥1 high-WW day + ≥1 rain event with recovery>2hr + duty imbalance>0.60
kpi039 = int(kpi034 >= 1 and any(re['KPI_029'] > 2.0 for re in rain_events) and kpi020 > 0.55)
print(f'KPI_039 Resilience Concern Flag: {kpi039}')

# KPI_040: Capacity/Control review flag
# Criteria: ≥1 pump with short-cycling days>1 + ≥1 rain uplift>1.5 + avg drawdown < 2 ft
avg_dd_all = statistics.mean(
    pump_daily[(d, p)]['KPI_014']
    for d in DATES for p in PUMP_IDS
    if pump_daily[(d, p)]['KPI_006'] > 0
)
kpi040 = int(any(v > 1 for v in kpi035.values()) and kpi036 >= 1 and avg_dd_all < 4.0)
print(f'KPI_040 Capacity/Control Review Flag: {kpi040} (avg drawdown={avg_dd_all:.2f}ft)')

KPI_033 Dry-weather P3 use: {'starts_per_day': 0.0, 'rt_per_day': 0.0}
KPI_034 High-WW days (>14.5ft): 1
KPI_035 Short-cycling days: {'P1': 0, 'P2': 0, 'P3': 0}
KPI_036 Rain stress events: 0
KPI_037 Trigger counts by severity: {'LOW': 9, 'HIGH': 1}
KPI_038 Trigger Persistence Index: 84
KPI_039 Resilience Concern Flag: 0
KPI_040 Capacity/Control Review Flag: 0 (avg drawdown=5.54ft)


## 9. Consolidated KPI Summary Table

In [10]:
# Summarise all 40 KPIs into a single table
kpi_summary = [
    # Station daily
    {'id':'KPI_001','name':'WetWell_Max_Daily',           'grain':'daily_station', 'value': round(max(daily[d]['KPI_001'] for d in DATES),2), 'units':'ft',       'tier':1},
    {'id':'KPI_002','name':'WetWell_P95_Daily',           'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_002'] for d in DATES),2), 'units':'ft', 'tier':2},
    {'id':'KPI_003','name':'WetWell_Mean_Daily',          'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_003'] for d in DATES),2), 'units':'ft', 'tier':3},
    {'id':'KPI_004','name':'Total_Starts_Daily',          'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_004'] for d in DATES),1), 'units':'starts/day','tier':1},
    {'id':'KPI_005','name':'Total_Runtime_Daily',         'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_005'] for d in DATES),1), 'units':'min/day',   'tier':2},
    # Pump daily (P1 representative)
    {'id':'KPI_006','name':'Pump_Starts_Daily (P1)',       'grain':'daily_pump',    'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_006'] for d in DATES),1), 'units':'starts/day','tier':2},
    {'id':'KPI_007','name':'Pump_Runtime_Daily (P1)',      'grain':'daily_pump',    'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_007'] for d in DATES),1), 'units':'min/day',   'tier':2},
    {'id':'KPI_008','name':'Pump_Median_Runtime (P1)',     'grain':'daily_pump',    'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_008'] for d in DATES if pump_daily[(d,'P1')]['KPI_006']>0),2), 'units':'min/run','tier':1},
    {'id':'KPI_009','name':'Pump_ShortRun_Fraction (P1)',  'grain':'daily_pump',    'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_009'] for d in DATES),4), 'units':'fraction','tier':1},
    {'id':'KPI_010','name':'Pump_LongRun_Fraction (P1)',   'grain':'daily_pump',    'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_010'] for d in DATES),4), 'units':'fraction','tier':3},
    {'id':'KPI_011','name':'Pump_Median_Current_A (P1)',   'grain':'daily_pump',    'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_011'] for d in DATES if pump_daily[(d,'P1')]['KPI_006']>0),2), 'units':'A','tier':2},
    {'id':'KPI_012','name':'Pump_Max_Current_A (P1)',      'grain':'daily_pump',    'value': round(max(pump_daily[(d,'P1')]['KPI_012'] for d in DATES),2), 'units':'A','tier':2},
    {'id':'KPI_013','name':'Pump_Running_Current_Ratio (P1)','grain':'daily_pump', 'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_013'] for d in DATES if pump_daily[(d,'P1')]['KPI_006']>0),4), 'units':'ratio','tier':1},
    {'id':'KPI_014','name':'Pump_Drawdown_Per_Run (P1)',   'grain':'run_event',     'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_014'] for d in DATES if pump_daily[(d,'P1')]['KPI_006']>0),3), 'units':'ft/run','tier':1},
    {'id':'KPI_015','name':'Pump_Drawdown_Rate (P1)',      'grain':'run_event',     'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_015'] for d in DATES if pump_daily[(d,'P1')]['KPI_006']>0),5), 'units':'ft/min','tier':2},
    {'id':'KPI_016','name':'Pump_Drawdown_Per_AmpMin (P1)','grain':'run_event',     'value': round(statistics.mean(pump_daily[(d,'P1')]['KPI_016'] for d in DATES if pump_daily[(d,'P1')]['KPI_006']>0),6), 'units':'ft/(A·min)','tier':3},
    {'id':'KPI_017','name':'Any_Pump_On_Fraction',         'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_017'] for d in DATES),4), 'units':'fraction','tier':3},
    {'id':'KPI_018','name':'TwoPumpsOn_Fraction',          'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_018'] for d in DATES),4), 'units':'fraction','tier':1},
    {'id':'KPI_019_P1','name':'Pump_Availability (P1)',    'grain':'monthly_pump',  'value': kpi019['P1'], 'units':'flag','tier':1},
    {'id':'KPI_019_P2','name':'Pump_Availability (P2)',    'grain':'monthly_pump',  'value': kpi019['P2'], 'units':'flag','tier':1},
    {'id':'KPI_019_P3','name':'Pump_Availability (P3)',    'grain':'monthly_pump',  'value': kpi019['P3'], 'units':'flag','tier':1},
    {'id':'KPI_020','name':'Duty_Imbalance_Ratio',         'grain':'weekly_station','value': kpi020, 'units':'ratio','tier':2},
    {'id':'KPI_021','name':'WetWell_Fill_Rate_OffPump',    'grain':'minute_station','value': round(statistics.mean(daily[d]['KPI_021'] for d in DATES),5), 'units':'ft/min','tier':2},
    {'id':'KPI_022','name':'Inferred_Inflow_OffPump',      'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_022'] for d in DATES),1), 'units':'gpm','tier':1},
    {'id':'KPI_023','name':'Inferred_Inflow_OffPump_P95',  'grain':'daily_station', 'value': round(statistics.mean(daily[d]['KPI_023'] for d in DATES),1), 'units':'gpm','tier':2},
    {'id':'KPI_024','name':'Rain_Total_Daily',             'grain':'daily_station', 'value': round(sum(daily[d]['KPI_024'] for d in DATES),3), 'units':'in/period','tier':2},
    # Rain event (RE_01)
    {'id':'KPI_025_E1','name':'Rain_Event_Total (RE_01)',    'grain':'rain_event', 'value': rain_events[0]['KPI_025'], 'units':'in/event','tier':2},
    {'id':'KPI_026_E1','name':'Rain_Response_Uplift_Starts (RE_01)','grain':'rain_event','value': rain_events[0]['KPI_026'], 'units':'ratio','tier':1},
    {'id':'KPI_027_E1','name':'Rain_Response_Uplift_Runtime (RE_01)','grain':'rain_event','value': rain_events[0]['KPI_027'], 'units':'ratio','tier':1},
    {'id':'KPI_028_E1','name':'Rain_Response_Lag (RE_01)',   'grain':'rain_event', 'value': rain_events[0]['KPI_028'], 'units':'min','tier':2},
    {'id':'KPI_029_E1','name':'Recovery_Time_After_Rain (RE_01)','grain':'rain_event','value': rain_events[0]['KPI_029'], 'units':'hr','tier':1},
    {'id':'KPI_030_E1','name':'Event_Peak_WetWell (RE_01)', 'grain':'rain_event', 'value': rain_events[0]['KPI_030'], 'units':'ft','tier':1},
    {'id':'KPI_031_E1','name':'Event_MultiPump_Dependency (RE_01)','grain':'rain_event','value': rain_events[0]['KPI_031'], 'units':'fraction','tier':2},
    {'id':'KPI_032_E1','name':'Event_AssistPump_Use (RE_01, starts)','grain':'rain_event','value': rain_events[0]['KPI_032_starts'], 'units':'starts/event','tier':2},
    {'id':'KPI_033','name':'DryWeather_AssistPump_Use',    'grain':'daily_pump',    'value': kpi033['starts_per_day'], 'units':'starts/day','tier':2},
    # Monthly
    {'id':'KPI_034','name':'HighWetWell_Days_Count',       'grain':'monthly_station','value': kpi034, 'units':'days','tier':2},
    {'id':'KPI_035_P1','name':'ShortCycling_Days_Count (P1)','grain':'monthly_pump','value': kpi035['P1'], 'units':'days','tier':2},
    {'id':'KPI_036','name':'RainStress_Events_Count',      'grain':'monthly_station','value': kpi036, 'units':'events','tier':2},
    {'id':'KPI_037_HIGH','name':'Trigger_Count HIGH',      'grain':'daily_station', 'value': kpi037.get('HIGH',0), 'units':'count','tier':3},
    {'id':'KPI_037_MED', 'name':'Trigger_Count MEDIUM',   'grain':'daily_station', 'value': kpi037.get('MEDIUM',0), 'units':'count','tier':3},
    {'id':'KPI_038','name':'Trigger_Persistence_Index',   'grain':'monthly_station','value': kpi038, 'units':'score','tier':3},
    {'id':'KPI_039','name':'Resilience_Concern_Flag',     'grain':'monthly_station','value': kpi039, 'units':'flag','tier':1},
    {'id':'KPI_040','name':'Capacity_Control_Review_Flag','grain':'monthly_station','value': kpi040, 'units':'flag','tier':1},
]

print(f"{'KPI_ID':<18} {'Name':<42} {'Value':>10} {'Units':<14} {'Tier'}")
print('-'*90)
for k in kpi_summary:
    print(f"{k['id']:<18} {k['name']:<42} {str(k['value']):>10} {k['units']:<14} T{k['tier']}")

KPI_ID             Name                                            Value Units          Tier
------------------------------------------------------------------------------------------
KPI_001            WetWell_Max_Daily                                18.0 ft             T1
KPI_002            WetWell_P95_Daily                               12.11 ft             T2
KPI_003            WetWell_Mean_Daily                               8.55 ft             T3
KPI_004            Total_Starts_Daily                               24.7 starts/day     T1
KPI_005            Total_Runtime_Daily                            1088.7 min/day        T2
KPI_006            Pump_Starts_Daily (P1)                           23.1 starts/day     T2
KPI_007            Pump_Runtime_Daily (P1)                         977.1 min/day        T2
KPI_008            Pump_Median_Runtime (P1)                        32.36 min/run        T1
KPI_009            Pump_ShortRun_Fraction (P1)                       0.0 fraction       

## 10. Export Dashboard JSON

In [11]:
# ── Time series (every 5 min for charting) ────────────────────────────────
ts_5min = []
for r in records[::5]:
    ts_5min.append({
        'ts': r['ts'], 'ww': r['ww_level_ft'],
        'P1': r['P1_on'], 'P2': r['P2_on'], 'P3': r['P3_on'],
        'n_on': r['n_pumps_on'],
        'rain': round(r['rain_in_per_min'] * 60, 4),   # in/hr for display
        'P1a': r['P1_amp'], 'P2a': r['P2_amp'], 'P3a': r['P3_amp'],
    })

# ── Hourly inflow estimate for chart ──────────────────────────────────────
hourly_inflow = []
for hr in range(SIM_DAYS * 24):
    start_m = hr * 60
    hr_recs = records[start_m:start_m+60]
    off_recs = [r for r in hr_recs if r['n_pumps_on'] == 0]
    if off_recs:
        dl_sum = sum(off_recs[i]['ww_level_ft'] - off_recs[i-1]['ww_level_ft']
                     for i in range(1, len(off_recs)) if off_recs[i]['ww_level_ft'] > off_recs[i-1]['ww_level_ft'])
        gpm = (WW_AREA_SQFT * dl_sum * GAL_PER_CUFT)
    else:
        gpm = 0
    ts_h = (T0 + datetime.timedelta(hours=hr)).strftime('%Y-%m-%d %H:%M')
    hourly_inflow.append({'ts': ts_h, 'inflow_gpm': round(gpm, 1)})

# ── Per-pump summary across all days ─────────────────────────────────────
pump_summary = []
for p in PUMP_IDS:
    p_runs = [e for e in run_events if e['pump'] == p]
    total_rt   = sum(e['duration_min'] for e in p_runs)
    all_durs   = [e['duration_min'] for e in p_runs]
    all_dds    = [e['start_level'] - e['end_level'] for e in p_runs]
    pump_summary.append({
        'pump': p, 'role': PUMP_ROLES[p], 'rating_a': PUMP_RATING_A[p],
        'total_starts': len(p_runs),
        'total_rt_min': total_rt,
        'mean_run_min': round(statistics.mean(all_durs), 2) if all_durs else 0,
        'median_run_min': round(statistics.median(all_durs), 2) if all_durs else 0,
        'short_run_frac': round(sum(1 for d in all_durs if d <= SHORT_RUN_THRESH) / max(1, len(all_durs)), 3),
        'long_run_frac':  round(sum(1 for d in all_durs if d >= LONG_RUN_THRESH)  / max(1, len(all_durs)), 3),
        'mean_drawdown_ft': round(statistics.mean(all_dds), 3) if all_dds else 0,
        'runtime_share': round(pump_rt_share[p], 3),
        'availability': kpi019[p],
    })

# ── Daily KPIs list ───────────────────────────────────────────────────────
daily_list = [
    {**daily[d],
     'P1_starts': pump_daily[(d,'P1')]['KPI_006'],
     'P2_starts': pump_daily[(d,'P2')]['KPI_006'],
     'P3_starts': pump_daily[(d,'P3')]['KPI_006'],
     'P1_rt_min': pump_daily[(d,'P1')]['KPI_007'],
     'P2_rt_min': pump_daily[(d,'P2')]['KPI_007'],
     'P3_rt_min': pump_daily[(d,'P3')]['KPI_007'],
     'P1_shortfrac': pump_daily[(d,'P1')]['KPI_009'],
     'P2_shortfrac': pump_daily[(d,'P2')]['KPI_009'],
     'P3_shortfrac': pump_daily[(d,'P3')]['KPI_009'],
    } for d in DATES
]

payload = {
    'meta': {
        'station': STATION_ID,
        'ww_area_sqft': WW_AREA_SQFT,
        'sim_days': SIM_DAYS,
        'n_minutes': N_MINUTES,
        'total_run_events': len(run_events),
        'rain_events_count': len(rain_events),
        'short_run_thresh_min': SHORT_RUN_THRESH,
        'long_run_thresh_min': LONG_RUN_THRESH,
        'stress_level_ft': STRESS_LEVEL_FT,
    },
    'kpi_summary': kpi_summary,
    'daily': daily_list,
    'pump_summary': pump_summary,
    'rain_events': rain_events,
    'ts_5min': ts_5min,
    'hourly_inflow': hourly_inflow,
    'triggers': triggers,
    'kpi037': kpi037,
    'kpi038': kpi038,
    'kpi039': kpi039,
    'kpi040': kpi040,
}

with open('scada_kpi_data.json', 'w') as f:
    json.dump(payload, f, indent=2, default=str)

print('Exported scada_kpi_data.json')
print(f'  KPI rows: {len(kpi_summary)}')
print(f'  5-min TS points: {len(ts_5min)}')
print(f'  Daily rows: {len(daily_list)}')

Exported scada_kpi_data.json
  KPI rows: 43
  5-min TS points: 2016
  Daily rows: 7
